# Phase 3 — Model Development and Hyperparameter Optimization

This notebook develops and optimizes predictive models using only the Cleveland development partition. Hyperparameter optimization is performed with stratified cross-validation on the training split, while the validation split is reserved for model selection, calibration, and threshold analysis. The Cleveland test set, together with the Hungarian and Swiss datasets, remains completely untouched to preserve an unbiased final evaluation in the subsequent phase.

No trained models or final evaluation artifacts are exported from this notebook.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target",
]
RAW_FEATURES = [c for c in COLUMN_NAMES if c != "target"]

SITE_FILES = {
    "cleveland": {
        "filename": "processed.cleveland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
    },
    "hungarian": {
        "filename": "processed.hungarian.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data",
    },
    "swiss": {
        "filename": "processed.switzerland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.switzerland.data",
    },
}

def ensure_raw_files():
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    paths = {}
    for site, info in SITE_FILES.items():
        path = RAW_DATA_DIR / info["filename"]
        paths[site] = path
        if not path.exists() or path.stat().st_size == 0:
            print(f"Downloading {site}...")
            urlretrieve(info["url"], path)
    return paths

def load_site(path, site):
    df = pd.read_csv(path, header=None, names=COLUMN_NAMES, na_values="?")
    for col in COLUMN_NAMES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["target_original"] = df["target"]
    df["target"] = (df["target"] > 0).astype(int)
    df["site"] = site
    df["original_index"] = df.index
    return df

def load_all_sites():
    paths = ensure_raw_files()
    return {site: load_site(path, site) for site, path in paths.items()}

def split_cleveland_v2(cleveland, random_state=RANDOM_STATE):
    # UCI processed Cleveland has no explicit patient ID. Therefore, rows are treated as independent patients.
    # If a patient_id column becomes available later, replace this with GroupShuffleSplit.
    cleveland = cleveland.copy()

    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=random_state)
    rest_idx, test_idx = next(sss_test.split(cleveland, cleveland["target"]))

    rest = cleveland.iloc[rest_idx].copy()
    test = cleveland.iloc[test_idx].copy()

    # Validation is 15% of total. After removing 15% test, validation is 15/85 of the remaining rows.
    val_fraction_of_rest = 0.15 / 0.85
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_fraction_of_rest, random_state=random_state)
    train_rel_idx, val_rel_idx = next(sss_val.split(rest, rest["target"]))

    train = rest.iloc[train_rel_idx].copy()
    validation = rest.iloc[val_rel_idx].copy()

    train["partition"] = "train"
    validation["partition"] = "validation"
    test["partition"] = "test"

    return train, validation, test

def verify_no_overlap(train, validation, test):
    sets = {
        "train": set(train["original_index"]),
        "validation": set(validation["original_index"]),
        "test": set(test["original_index"]),
    }
    assert sets["train"].isdisjoint(sets["validation"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["validation"].isdisjoint(sets["test"])
    print("No Cleveland original_index overlap across train/validation/test.")

def load_phase1_v2_in_memory():
    sites = load_all_sites()
    train, validation, test = split_cleveland_v2(sites["cleveland"])
    verify_no_overlap(train, validation, test)
    return {
        "cleveland_full": sites["cleveland"],
        "cleveland_train": train,
        "cleveland_validation": validation,
        "cleveland_test": test,
        "hungarian": sites["hungarian"],
        "swiss": sites["swiss"],
    }

In [2]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

import optuna
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
XGB_OPTUNA_TRIALS = 50

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "f1": "f1",
    "sensitivity": "recall",
    "specificity": make_scorer(specificity_score),
}
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

In [3]:
class ClinicalFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_df = self._to_dataframe(X)
        self.feature_names_in_ = list(X_df.columns)
        self.feature_names_out_ = list(self.transform(X_df).columns)
        return self

    def transform(self, X):
        X_df = self._to_dataframe(X).copy()
        for col in X_df.columns:
            X_df[col] = pd.to_numeric(X_df[col], errors="coerce")
        X_df["age_x_thalach"] = X_df["age"] * X_df["thalach"]
        X_df["age_x_oldpeak"] = X_df["age"] * X_df["oldpeak"]
        X_df["trestbps_x_chol"] = X_df["trestbps"] * X_df["chol"]
        X_df["cp_x_thal"] = X_df["cp"] * X_df["thal"]
        X_df["exang_x_oldpeak"] = X_df["exang"] * X_df["oldpeak"]
        X_df["framingham_proxy"] = (
            X_df["age"] / 10.0
            + X_df["sex"] * 2.0
            + (X_df["chol"] - 200.0) / 40.0
            + (X_df["trestbps"] - 120.0) / 20.0
            + X_df["fbs"] * 1.5
        )
        return X_df

    def get_feature_names_out(self, input_features=None):
        return np.array(getattr(self, "feature_names_out_", input_features if input_features is not None else []))

    @staticmethod
    def _to_dataframe(X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=RAW_FEATURES[: X.shape[1]])

def make_preprocessor():
    return Pipeline([
        ("features", ClinicalFeatureEngineer()),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

class CORALAdapter:
    def __init__(self, eps=1e-5):
        self.eps = eps
    def fit(self, X_source, X_target):
        Xs = np.asarray(X_source, dtype=float)
        Xt = np.asarray(X_target, dtype=float)
        self.source_mean_ = Xs.mean(axis=0)
        self.target_mean_ = Xt.mean(axis=0)
        Cs = np.cov(Xs, rowvar=False) + self.eps * np.eye(Xs.shape[1])
        Ct = np.cov(Xt, rowvar=False) + self.eps * np.eye(Xt.shape[1])
        self.source_transform_ = self._mpower(Cs, -0.5)
        self.target_transform_ = self._mpower(Ct, 0.5)
        return self
    def transform_source_to_target(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.source_mean_) @ self.source_transform_ @ self.target_transform_ + self.target_mean_
    @staticmethod
    def _mpower(matrix, power):
        vals, vecs = np.linalg.eigh(matrix)
        vals = np.maximum(vals, 1e-12)
        return vecs @ np.diag(vals ** power) @ vecs.T

def summarize_cv(name, results):
    row = {"model": name}
    for metric in scoring:
        values = results[f"test_{metric}"]
        row[f"{metric}_mean"] = float(np.mean(values))
        row[f"{metric}_std"] = float(np.std(values))
    return row

In [4]:
phase1 = load_phase1_v2_in_memory()
train = phase1["cleveland_train"].copy()
validation = phase1["cleveland_validation"].copy()
cleveland_test = phase1["cleveland_test"].copy()
hungarian = phase1["hungarian"].copy()
swiss = phase1["swiss"].copy()

X_train = train[RAW_FEATURES]
y_train = train["target"]
X_val = validation[RAW_FEATURES]
y_val = validation["target"]

print("Training rows:", X_train.shape)
print("Validation rows:", X_val.shape)
print("Locked test rows:", cleveland_test.shape)
print("External rows:", hungarian.shape, swiss.shape)

No Cleveland original_index overlap across train/validation/test.
Training rows: (211, 13)
Validation rows: (46, 13)
Locked test rows: (46, 18)
External rows: (294, 17) (123, 17)


## Feature Engineering and Preprocessing

A unified preprocessing pipeline is applied to every candidate model to ensure a fair comparison. The workflow consists of domain-informed clinical feature engineering, median imputation for missing values, and feature standardization. All preprocessing components are fitted exclusively within the cross-validation folds of the Cleveland training data, preventing information leakage.

In [5]:
preview = ClinicalFeatureEngineer().fit_transform(X_train.head())
print("Raw features:", len(RAW_FEATURES))
print("Engineered features:", preview.shape[1])
preview

Raw features: 13
Engineered features: 19


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,age_x_thalach,age_x_oldpeak,trestbps_x_chol,cp_x_thal,exang_x_oldpeak,framingham_proxy
255,42.0,0.0,3.0,120.0,209.0,0.0,0.0,173.0,0.0,0.0,2.0,0.0,3.0,7266.0,0.0,25080.0,9.0,0.0,4.425
164,48.0,1.0,3.0,124.0,255.0,1.0,0.0,175.0,0.0,0.0,1.0,2.0,3.0,8400.0,0.0,31620.0,9.0,0.0,9.875
221,54.0,0.0,3.0,108.0,267.0,0.0,2.0,167.0,0.0,0.0,1.0,0.0,3.0,9018.0,0.0,28836.0,9.0,0.0,6.475
210,37.0,0.0,3.0,120.0,215.0,0.0,0.0,170.0,0.0,0.0,1.0,0.0,3.0,6290.0,0.0,25800.0,9.0,0.0,4.075
38,55.0,1.0,4.0,132.0,353.0,0.0,0.0,132.0,1.0,1.2,2.0,1.0,7.0,7260.0,66.0,46596.0,28.0,1.2,11.925


## Logistic Regression

Logistic Regression serves as the primary interpretable baseline. A comprehensive grid search is used to optimize the regularization strategy, solver configuration, and class-weighting scheme using stratified cross-validation.

In [6]:
lr_pipeline = Pipeline([
    ("preprocess", make_preprocessor()),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

lr_grid = [
    {"model__solver": ["liblinear"], "model__penalty": ["l1", "l2"], "model__C": [0.01, 0.05, 0.1, 0.5, 1, 2, 5], "model__class_weight": [None, "balanced"]},
    {"model__solver": ["lbfgs"], "model__penalty": ["l2"], "model__C": [0.01, 0.05, 0.1, 0.5, 1, 2, 5], "model__class_weight": [None, "balanced"]},
]

lr_search = GridSearchCV(lr_pipeline, lr_grid, scoring="roc_auc", cv=cv, n_jobs=-1, refit=True)
lr_search.fit(X_train, y_train)
print("Best LR AUC:", lr_search.best_score_)
print("Best LR params:", lr_search.best_params_)
lr_summary = summarize_cv("Logistic Regression", cross_validate(lr_search.best_estimator_, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1))
lr_summary

Best LR AUC: 0.9180018722696068
Best LR params: {'model__C': 0.5, 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'liblinear'}


{'model': 'Logistic Regression',
 'roc_auc_mean': 0.9180018722696068,
 'roc_auc_std': 0.013837522522751245,
 'average_precision_mean': 0.9148219540498921,
 'average_precision_std': 0.014067967842797333,
 'f1_mean': 0.8141969777263893,
 'f1_std': 0.04239992568143711,
 'sensitivity_mean': 0.8026315789473685,
 'sensitivity_std': 0.09125168385347297,
 'specificity_mean': 0.858893280632411,
 'specificity_std': 0.05381303533633993}

## Extreme Gradient Boosting (XGBoost)

An XGBoost classifier is optimized using Optuna-based hyperparameter search. The optimization process explores a broad parameter space over multiple trials to identify a high-performing tree-based ensemble while maintaining a reproducible experimental protocol.

In [7]:
def make_xgb_pipeline(params):
    return Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist", **params)),
    ])

def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 450),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5, 2.5),
    }
    return cross_validate(make_xgb_pipeline(params), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)["test_score"].mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective_xgb, n_trials=XGB_OPTUNA_TRIALS, show_progress_bar=True)
assert len(study.trials) >= 50
print("Best XGBoost AUC:", study.best_value)
print("Best XGBoost params:", study.best_params)
xgb_best = make_xgb_pipeline(study.best_params)
xgb_summary = summarize_cv("XGBoost + Optuna", cross_validate(xgb_best, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1))
xgb_summary

  0%|          | 0/50 [00:00<?, ?it/s]

Best XGBoost AUC: 0.9172238402329935
Best XGBoost params: {'n_estimators': 298, 'max_depth': 3, 'learning_rate': 0.010131396599536682, 'subsample': 0.9734095487296471, 'colsample_bytree': 0.7972159874326937, 'min_child_weight': 5.708324149699832, 'gamma': 4.550200187953906, 'reg_alpha': 0.014564331587762814, 'reg_lambda': 0.05988996831698874, 'scale_pos_weight': 0.844977726025752}


{'model': 'XGBoost + Optuna',
 'roc_auc_mean': 0.9172238402329935,
 'roc_auc_std': 0.018098070095764116,
 'average_precision_mean': 0.9251622477597191,
 'average_precision_std': 0.01143811324681806,
 'f1_mean': 0.8404516358463727,
 'f1_std': 0.045060828322178614,
 'sensitivity_mean': 0.7936842105263158,
 'sensitivity_std': 0.07388318998102919,
 'specificity_mean': 0.9213438735177866,
 'specificity_std': 0.05043825182908862}

## Multi-Layer Perceptron (MLP)

A compact feed-forward neural network is evaluated as the nonlinear baseline. Hyperparameters are optimized using randomized search, and class imbalance is addressed through balanced sample weighting during training.

In [8]:
class WeightedMLPClassifier(MLPClassifier):
    def fit(self, X, y):
        sample_weight = compute_sample_weight(class_weight="balanced", y=y)
        try:
            return super().fit(X, y, sample_weight=sample_weight)
        except TypeError:
            return super().fit(X, y)

mlp_pipeline = Pipeline([
    ("preprocess", make_preprocessor()),
    ("model", WeightedMLPClassifier(max_iter=1200, early_stopping=True, validation_fraction=0.20, n_iter_no_change=40, random_state=RANDOM_STATE)),
])

mlp_grid = {
    "model__hidden_layer_sizes": [(8,), (16,), (32,), (16, 8), (32, 16)],
    "model__activation": ["relu", "tanh"],
    "model__solver": ["adam"],
    "model__alpha": [1e-4, 1e-3, 1e-2, 1e-1],
    "model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    "model__batch_size": [16, 32, 64],
}

mlp_search = RandomizedSearchCV(mlp_pipeline, mlp_grid, n_iter=35, scoring="roc_auc", cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True)
mlp_search.fit(X_train, y_train)
print("Best MLP AUC:", mlp_search.best_score_)
print("Best MLP params:", mlp_search.best_params_)
mlp_summary = summarize_cv("Small MLP", cross_validate(mlp_search.best_estimator_, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1))
mlp_summary

Best MLP AUC: 0.9197014770126899
Best MLP params: {'model__solver': 'adam', 'model__learning_rate_init': 0.005, 'model__hidden_layer_sizes': (32,), 'model__batch_size': 16, 'model__alpha': 0.01, 'model__activation': 'relu'}


{'model': 'Small MLP',
 'roc_auc_mean': 0.9197014770126899,
 'roc_auc_std': 0.022385744507065188,
 'average_precision_mean': 0.9189549470382448,
 'average_precision_std': 0.017682209294583338,
 'f1_mean': 0.7785448204803045,
 'f1_std': 0.07652903113003928,
 'sensitivity_mean': 0.721578947368421,
 'sensitivity_std': 0.15397233265874985,
 'specificity_mean': 0.9019762845849802,
 'specificity_std': 0.09311300004253968}

## Cross-Validation Performance Comparison

The optimized candidate models are compared using identical cross-validation splits on the Cleveland training set. This comparison provides an unbiased basis for selecting the strongest model before any evaluation on the held-out datasets.

In [9]:
cv_summary = pd.DataFrame([lr_summary, xgb_summary, mlp_summary]).sort_values("roc_auc_mean", ascending=False)
cv_summary

,model,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,f1_mean,f1_std,sensitivity_mean,sensitivity_std,specificity_mean,specificity_std
2,Small MLP,0.919701,0.022386,0.918955,0.017682,0.778545,0.076529,0.721579,0.153972,0.901976,0.093113
0,Logistic Regression,0.918002,0.013838,0.914822,0.014068,0.814197,0.042400,0.802632,0.091252,0.858893,0.053813
1,XGBoost + Optuna,0.917224,0.018098,0.925162,0.011438,0.840452,0.045061,0.793684,0.073883,0.921344,0.050438


## Domain Adaptation Sanity Check

A preliminary verification of the CORAL domain adaptation module is performed to confirm that feature alignment operates as expected. The method is intentionally excluded from model selection and will instead be evaluated systematically during the final generalization experiments in Phase 4.

In [10]:
pre = make_preprocessor()
Xs = pre.fit_transform(X_train)
Xt = pre.transform(hungarian[RAW_FEATURES])
coral = CORALAdapter().fit(Xs, Xt)
print("Source processed:", Xs.shape)
print("Target processed:", Xt.shape)
print("Aligned source:", coral.transform_source_to_target(Xs).shape)

Source processed: (211, 19)
Target processed: (294, 19)
Aligned source: (211, 19)
